In [14]:
from typing import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [15]:
load_dotenv()

True

In [16]:
llm = ChatOpenAI(model="gpt-4.1-mini")

In [17]:
# define the state
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[str], add_messages]

In [18]:
# define the chatnode function
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [19]:
# 3. Build the graph: START -> chat -> END
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [27]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [28]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='15e52e7c-769d-4e0e-9f2b-1fa4bf3c7c03'),
  AIMessage(content='Not approved.', additional_kwargs={}, response_metadata={}, id='2f30aeb9-5327-491d-9c52-abde50532aab', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='256bfa12-a77f-4f46-881a-7deee9d00baa')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='227d3169e6d64ff06f60e84cc451c2c6')]}

In [29]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [30]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [31]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [32]:
print(final_result["messages"][-1].content)

Sure! Imagine you are standing on a hill and want to find the lowest point in the valley nearby. Since it might be foggy and you can't see far, you decide to take small steps going downhill.

Gradient descent is like that: it’s a way a computer learns by taking small steps in the direction that makes a mistake smaller, until it finds the best solution. Each step uses information about the slope (called the gradient) to know which way is downhill. Over time, these steps lead the computer to the lowest point, or the best answer.
